In [4]:
!pip install rouge

In [5]:
import os
import time
import torch
import mlflow
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge  # pip install rouge   (lightweight NLTK-friendly alternative)

device = "cuda" if torch.cuda.is_available() else "cpu"

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("flickr8k-blip-finetuning")

IMAGES_DIR = r"C:\Users\Lenovo\Internship\data\Flickr8k\images"
MODEL_NAME = "Salesforce/blip-image-captioning-base"

In [15]:
import pandas as pd

captions_df = pd.read_csv(
    r"C:\Users\Lenovo\Internship\data\Flickr8k\captions.txt",
    sep="|"
)

print(captions_df.shape)
print(captions_df.columns.tolist())
captions_df.head()

(40455, 3)
['image_name', 'caption_number', 'caption_text']


,image_name,caption_number,caption_text
0,1000268201_693b08cb0e.jpg,0,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,1,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,2,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,3,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,4,A little girl in a pink dress going into a woo...


In [16]:
from sklearn.model_selection import train_test_split

unique_images = captions_df["image_name"].unique()
train_images, val_images = train_test_split(unique_images, test_size=0.1, random_state=42)

train_df = captions_df[captions_df["image_name"].isin(train_images)].reset_index(drop=True)
val_df = captions_df[captions_df["image_name"].isin(val_images)].reset_index(drop=True)

print(f"Train images: {len(train_images)} | Train rows: {len(train_df)}")
print(f"Val images: {len(val_images)} | Val rows: {len(val_df)}")

Train images: 7281 | Train rows: 36405
Val images: 810 | Val rows: 4050


In [6]:
class FlickrCaptionDataset(Dataset):
    def __init__(self, df, images_dir, processor):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.images_dir, row["image_name"])).convert("RGB")
        encoding = self.processor(
            images=image,
            text=row["caption_text"],
            padding="max_length",
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in encoding.items()}

In [7]:
def load_model_for_finetuning():
    processor = BlipProcessor.from_pretrained(MODEL_NAME)
    model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

    # Freeze vision encoder, fine-tune only the text decoder
    for param in model.vision_model.parameters():
        param.requires_grad = False

    return model, processor

In [8]:
def fine_tune(train_df, subset_size, learning_rate, epochs=1, batch_size=8):
    subset_df = train_df.sample(n=min(subset_size, len(train_df)), random_state=42)

    model, processor = load_model_for_finetuning()
    dataset = FlickrCaptionDataset(subset_df, IMAGES_DIR, processor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate
    )

    model.train()
    total_loss = 0
    for epoch in range(epochs):
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=batch["input_ids"])
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    avg_loss = total_loss / (len(loader) * epochs)
    return model, processor, avg_loss

In [9]:
def generate_captions(model, processor, df, images_dir, decoding="greedy", num_beams=1, max_new_tokens=25):
    model.eval()
    predictions, references = [], []

    for _, row in df.iterrows():
        image = Image.open(os.path.join(images_dir, row["image_name"])).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)

        with torch.no_grad():
            if decoding == "beam":
                out = model.generate(**inputs, num_beams=num_beams, max_new_tokens=max_new_tokens)
            else:
                out = model.generate(**inputs, max_new_tokens=max_new_tokens)

        caption = processor.decode(out[0], skip_special_tokens=True)
        predictions.append(caption)
        references.append(row["caption_text"])

    return predictions, references

In [17]:
import gc

def run_experiment(run_name, subset_size, learning_rate, decoding, num_beams=1,
                    val_sample_size=100, epochs=1):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "subset_size": subset_size,
            "learning_rate": learning_rate,
            "decoding": decoding,
            "num_beams": num_beams,
            "epochs": epochs
        })

        start = time.time()
        model, processor, train_loss = fine_tune(
            train_df, subset_size, learning_rate, epochs=epochs
        )
        train_time = time.time() - start

        val_sample = val_df.sample(n=min(val_sample_size, len(val_df)), random_state=42)
        preds, refs = generate_captions(
            model, processor, val_sample, IMAGES_DIR,
            decoding=decoding, num_beams=num_beams
        )
        metrics = evaluate_captions(preds, refs)

        mlflow.log_metric("train_loss", train_loss)
        mlflow.log_metric("train_time_sec", train_time)
        mlflow.log_metrics(metrics)

        print(f"[{run_name}] loss={train_loss:.4f} | BLEU={metrics['bleu']:.4f} | ROUGE-L={metrics['rougeL']:.4f}")

    # --- cleanup to free memory before next run ---
    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

In [18]:
run_experiment("run1_lr5e-5_greedy", subset_size=500, learning_rate=5e-5, decoding="greedy")

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[run1_lr5e-5_greedy] loss=4.5117 | BLEU=0.0783 | ROUGE-L=0.3753


{'bleu': np.float64(0.07825052107805497),
 'rougeL': np.float64(0.37530170687420505)}

In [19]:
run_experiment("run2_lr1e-5_beam3", subset_size=500, learning_rate=1e-5, decoding="beam", num_beams=3)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[run2_lr1e-5_beam3] loss=6.4142 | BLEU=0.0918 | ROUGE-L=0.3847


{'bleu': np.float64(0.09180927235128589),
 'rougeL': np.float64(0.38474313109476205)}

In [20]:
run_experiment("run3_lr5e-5_subset1000_beam3", subset_size=1000, learning_rate=5e-5, decoding="beam", num_beams=3)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[run3_lr5e-5_subset1000_beam3] loss=2.7740 | BLEU=0.0931 | ROUGE-L=0.3972


{'bleu': np.float64(0.09312769799263881),
 'rougeL': np.float64(0.3971737824538111)}

In [22]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("flickr8k-blip-finetuning")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

comparison = runs[[
    "tags.mlflow.runName", "params.subset_size", "params.learning_rate",
    "params.decoding", "params.num_beams",
    "metrics.bleu", "metrics.rougeL", "metrics.train_loss", "metrics.train_time_sec"
]].sort_values("metrics.bleu", ascending=False)

comparison

,tags.mlflow.runName,params.subset_size,params.learning_rate,params.decoding,params.num_beams,metrics.bleu,metrics.rougeL,metrics.train_loss,metrics.train_time_sec
0,run3_lr5e-5_subset1000_beam3,1000,5e-05,beam,3,0.093128,0.397174,2.774050,2488.608114
1,run2_lr1e-5_beam3,500,1e-05,beam,3,0.091809,0.384743,6.414196,1350.248449
2,run1_lr5e-5_greedy,500,5e-05,greedy,1,0.078251,0.375302,4.511695,1190.254336
3,run1_lr5e-5_greedy,500,5e-05,greedy,1,NaN,NaN,NaN,NaN
4,run1_lr5e-5_greedy,500,5e-05,greedy,1,NaN,NaN,NaN,NaN
5,run1_lr5e-5_greedy,500,5e-05,greedy,1,NaN,NaN,NaN,NaN


## Day 4 — Fine-tuning Experiment Comparison

Three configurations were fine-tuned and evaluated on the Flickr8k validation subset:

- **Run 3** (lr=5e-5, subset=1000, beam search, beams=3) achieved the best performance: BLEU=0.093, ROUGE-L=0.397, with the lowest training loss (2.77). Doubling the training subset size compared to runs 1 and 2 appears to have driven the biggest improvement.
- **Run 2** (lr=1e-5, subset=500, beam search, beams=3) scored close behind (BLEU=0.092, ROUGE-L=0.385) despite a 5x lower learning rate and half the data — suggesting beam search decoding itself is a stronger lever than learning rate in this setup.
- **Run 1** (lr=5e-5, subset=500, greedy decoding) performed worst (BLEU=0.078, ROUGE-L=0.375), reinforcing that **beam search outperforms greedy decoding** at the same learning rate.

**Conclusion**: Run 3's configuration (lr=5e-5, larger subset, beam search) will be selected as the baseline going into Day 5, though the strong performance of run2 suggests decoding strategy (beam vs. greedy) has a larger marginal effect than learning rate alone.